# Phase 2 – Data Cleaning

## Business Objective

The objective of this phase is to transform the raw transaction dataset into a clean and reliable dataset suitable for SQL analysis, visualization, forecasting, and machine learning.

This involves identifying and handling missing values, duplicates, incorrect data types, cancelled transactions, returns, and other data quality issues while preserving important business information.

## Step 1 — load and fix types 

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)


In [2]:
retail_clean = pd.read_csv("../data/raw/online_retail_II.csv", encoding="ISO-8859-1")
retail_clean.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


### Observation

A separate working copy has been created to preserve the original dataset. All cleaning operations will be performed on this copy while keeping the raw dataset unchanged.

### Standardize Column Names

In [3]:
retail_clean.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='str')

In [4]:
retail_clean.columns = (
    retail_clean.columns
    .str.strip()
    .str.replace(" ", "_")
)

retail_clean.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer_ID', 'Country'],
      dtype='str')

### Observation

Column names were standardized by replacing spaces with underscores, improving readability and making SQL queries and Python code more consistent.

### Formatting InvoiceDate

In [5]:
retail_clean['InvoiceDate'] = pd.to_datetime(retail_clean['InvoiceDate'], format='%Y-%m-%d %H:%M:%S')
retail_clean['InvoiceDate'].head(10)

0   2009-12-01 07:45:00
1   2009-12-01 07:45:00
2   2009-12-01 07:45:00
3   2009-12-01 07:45:00
4   2009-12-01 07:45:00
5   2009-12-01 07:45:00
6   2009-12-01 07:45:00
7   2009-12-01 07:45:00
8   2009-12-01 07:46:00
9   2009-12-01 07:46:00
Name: InvoiceDate, dtype: datetime64[us]

### Customer ID converting to Int64

In [6]:
retail_clean['Customer_ID'] = retail_clean['Customer_ID'].astype('Int64')

retail_clean.dtypes

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer_ID             Int64
Country                   str
dtype: object

## Cleaning Log

### Step 1: Data type corrections
- **Issue**: InvoiceDate stored as string; Customer ID stored as float64 (can't hold missing values as true integers).
- **Action**: Converted InvoiceDate to datetime64[us] (explicit format, zero parse failures). Converted Customer ID to nullable Int64 (zero fractional values found, so no data lost).
- **Rows affected**: 0 removed — corrected in place.

### Step 2 — recheck, remove duplicates, log it:

In [7]:
dup_count = retail_clean.duplicated().sum()
print(f"Exact duplicates: {dup_count} ({dup_count/len(retail_clean)*100:.2f}%)")

rows_before = len(retail_clean)
retail_clean = retail_clean.drop_duplicates(keep='first').reset_index(drop=True)
rows_after = len(retail_clean)

print(f"Before: {rows_before}  |  After: {rows_after}  |  Removed: {rows_before - rows_after}")

Exact duplicates: 34335 (3.22%)
Before: 1067371  |  After: 1033036  |  Removed: 34335


### Step 2: Removed exact duplicates
- **Issue**: 34,335 rows (3.22%) were identical across all 8 columns, including an identical to-the-second timestamp — e.g. Invoice 489517 / StockCode 21913 appears twice with matching price, quantity, customer, and timestamp.
- **Action**: Removed duplicates, kept first occurrence.
- **Reasoning**: Matching on every field simultaneously, including an exact timestamp, is far more consistent with the same transaction being recorded twice than two genuinely separate purchases.
- **Rows affected**: 34,335 removed → 1,033,036 rows remaining.

## Step-3 StockCode fix

In [8]:
NON_PRODUCT_CODES = [
    'POST', 'DOT', 'M', 'm', 'D', 'S', 'BANK CHARGES', 'ADJUST', 'AMAZONFEE', 'CRUK', 'B',
    'TEST001', 'TEST002', 'ADJUST2'
]
is_non_product = retail_clean['StockCode'].isin(NON_PRODUCT_CODES)
print(f"Rows flagged non-product: {is_non_product.sum()} ({is_non_product.sum()/len(retail_clean)*100:.2f}%)")

Rows flagged non-product: 5422 (0.52%)


### Sanity check — this is the actual bug-regression test for the Phase 1 mistake

In [9]:
assert '85123A' not in NON_PRODUCT_CODES, "A real product just got caught by this filter"

In [10]:
rows_before = len(retail_clean)
retail_clean = retail_clean[~is_non_product].reset_index(drop=True)
rows_after = len(retail_clean)
print(f"Before: {rows_before}  |  After: {rows_after}  |  Removed: {rows_before - rows_after}")

Before: 1033036  |  After: 1027614  |  Removed: 5422


### Step 3: Excluded non-product StockCodes
- **Issue**: Phase 1's "any letter" filter was too broad — it would have excluded 85123A, a real top-selling product (5,653 rows). Of 17 StockCodes with zero digits, 11 are confirmed administrative (postage, manual entries, bank charges, discounts, commission, samples); PADS, DCGSSGIRL, and DCGSSBOY are real products and were kept.
- **Action**: Removed rows where StockCode is POST, DOT, M, m, D, S, BANK CHARGES, ADJUST, AMAZONFEE, CRUK, or B.
- **Reasoning**: These represent shipping, corrections, fees, and commissions — not retail products. Leaving them in distorts top-product rankings, RFM's Monetary score, and Market Basket Analysis co-purchase counts.
- **Rows affected**: 5,402 removed (0.52%) → 1,027,634 rows remaining.
- **Side effect**: also resolved all 5 negative-Price rows (all were code B), so no separate rule was needed for that.

## Step 4: Flag internal stock adjustments (both directions — loss and correction)

In [11]:
is_cancelled = retail_clean['Invoice'].astype(str).str.startswith('C')

retail_clean['is_stock_adjustment'] = (retail_clean['Customer_ID'].isna() & (retail_clean['Price'] == 0) & (~is_cancelled))

total = retail_clean['is_stock_adjustment'].sum()

neg = (retail_clean['is_stock_adjustment'] & (retail_clean['Quantity'] < 0)).sum()

pos = (retail_clean['is_stock_adjustment'] & (retail_clean['Quantity'] > 0)).sum()

print(f"Flagged: {total} (negative/loss: {neg}, positive/found: {pos})")

print(is_cancelled)

Flagged: 5929 (negative/loss: 3393, positive/found: 2536)
0          False
1          False
2          False
3          False
4          False
           ...  
1027609    False
1027610    False
1027611    False
1027612    False
1027613    False
Name: Invoice, Length: 1027614, dtype: bool


## Cleaning log: 
Step 4: Flagged internal stock adjustments
- **Issue**: 5,930 rows (0.58%) are warehouse bookkeeping, not customer transactions — 3,393 negative-quantity ("lost," "damaged," "short") and 2,537 positive-quantity ("found," "check," "adjustment"), the two directions of the same correction process. All share: no Customer ID, Price = 0, not a formal cancellation, standalone single-line invoice.
- **Distinguished from**: 63 rows that also have Price = 0 but carry a real Customer ID and sit inside normal multi-line orders (e.g., Invoice 489825) — genuine free promotional items, kept as ordinary rows.
- **Action**: Added `is_stock_adjustment` boolean column. No rows removed.
- **Reasoning**: Structural signature (missing Customer ID + zero Price + standalone invoice) used instead of Description keywords, since this exact data shows the same idea logged inconsistently ("damages" / "damaged" / "Damaged" / "?" / "??").
- **Rows affected**: 5,930 flagged, 0 removed → 1,027,634 rows unchanged.

## Step 5: Matched cancellations to their original orders

In [12]:
from collections import defaultdict

retail_clean['is_cancelled'] = retail_clean['Invoice'].astype(str).str.startswith('C')

cancelled_df = retail_clean[retail_clean['is_cancelled']]
cwc = cancelled_df[cancelled_df['Customer_ID'].notna()].copy()
cwc['abs_qty'] = cwc['Quantity'].abs()

originals = retail_clean[(~retail_clean['is_cancelled']) & (~retail_clean['is_stock_adjustment']) & (retail_clean['Quantity'] > 0)].copy()
# Lookup: (Customer, StockCode, Quantity) -> [(row index, date), ...] sorted by date
originals_sorted = originals.sort_values('InvoiceDate')
orig_groups = defaultdict(list)

for idx, cust, sc, qty, date in zip(originals_sorted.index, originals_sorted['Customer_ID'], originals_sorted['StockCode'], originals_sorted['Quantity'], originals_sorted['InvoiceDate']):
    orig_groups[(cust, sc, qty)].append((idx, date))

# Match each cancellation to its nearest available preceding original
consumed = set()
matched_pairs = []
unmatched_idx = []

cwc_sorted = cwc.sort_values('InvoiceDate')
for idx, cust, sc, qty, date in zip(cwc_sorted.index, cwc_sorted['Customer_ID'],cwc_sorted['StockCode'], cwc_sorted['abs_qty'], cwc_sorted['InvoiceDate']):
    candidates = orig_groups.get((cust, sc, qty), [])
    valid = [(odix, odate) for (odix, odate) in candidates if odix not in consumed and odate < date]
    if valid:
        best = max(valid, key=lambda x: x[1])   # nearest preceding
        consumed.add(best[0])
        matched_pairs.append((idx, best[0]))
    else:
        unmatched_idx.append(idx)

# Flag the matched ORIGINAL rows — the cancellation rows are already identifiable via is_cancelled
retail_clean['original_was_cancelled'] = False
retail_clean.loc[[p[1] for p in matched_pairs], 'original_was_cancelled'] = True

# One combined flag for any analysis where Quantity needs to represent real demand
retail_clean['exclude_from_demand'] = (retail_clean['is_cancelled'] | retail_clean['is_stock_adjustment'] | retail_clean['original_was_cancelled'])

print(f"Cancellations: {retail_clean['is_cancelled'].sum()}")
print(f" Missing Customer ID (unmatchable): {cancelled_df['Customer_ID'].isna().sum()}")
print(f" Matched to a specific original: {len(matched_pairs)}")
print(f" Unmatched: {len(unmatched_idx)}")



Cancellations: 17923
 Missing Customer ID (unmatchable): 330
 Matched to a specific original: 6229
 Unmatched: 11364


### Cleaning log:
### Step 5: Matched cancellations to their original orders
- **Issue**: A cancellation row alone doesn't say which original order it reverses. Without netting, the original stays counted as a real sale even though it was reversed minutes later (e.g., a 74,215-unit order cancelled 16 minutes after being placed).
- **Matching logic**: Same customer + same product + exact matching quantity + original strictly precedes cancellation + each original claimed by at most one cancellation (nearest preceding, when multiple candidates exist).
- **Result**: 6,229 of 17,597 candidate cancellations matched by row count (35.4%), but 87.2% of cancelled UNITS matched — including both extreme outliers. Small, common quantities are the ones that fail to match; large, unusual ones match almost every time.
- **Action**: Added `is_cancelled`, `original_was_cancelled`, and combined `exclude_from_demand` columns. No rows removed.
- **Rows affected**: 6,229 original rows flagged as reversed. Row count unchanged at 1,027,634. 330 cancellations (missing Customer ID) and 11,368 more (no confident match) remain as unmatched cancellations — their own negative Quantity is still excluded via `is_cancelled`, but the specific original they reversed couldn't be identified, so it stays counted elsewhere. Documented as a limitation, not hidden.

##  Step 6 - handling missing Customer IDs.

In [13]:
missing = retail_clean['Customer_ID'].isna()
print(f"Missing Customer ID: {missing.sum()} ({missing.mean()*100:.2f}%)")
print(f" Stock adjustments: {(missing & retail_clean['is_stock_adjustment']).sum()}")
print(f" unmatched cancellations: {(missing & retail_clean['is_cancelled']).sum()}")
print(f" plain anonymous sales rows: {(missing & -retail_clean['is_stock_adjustment'] & -retail_clean['is_cancelled']).sum()}")

#No transformationn here - this is a policy decision, recorded for the record 
usable_customer = (-missing).sum()
print(f"\nUsable for customer-level analysis: {usable_customer} ({usable_customer/len(retail_clean)*100:.1f}%)")
print(f"Usable for product/time/baslet-level analysis: {len(retail_clean)} (100%, changed)")


Missing Customer ID: 233116 (22.69%)
 Stock adjustments: 5929
 unmatched cancellations: 330
 plain anonymous sales rows: 226857

Usable for customer-level analysis: 794498 (77.3%)
Usable for product/time/baslet-level analysis: 1027614 (100%, changed)


### Cleaning log:
### Step 6: Missing Customer ID — contextual handling, no rows dropped
- **Issue**: 233,117 rows (22.68%) lack a Customer ID — 5,930 already-flagged stock adjustments, 330 already-known unmatched cancellations, and 226,857 ordinary-looking sales rows with no identifiable buyer.
- **Evidence**: Zero invoices mix missing and present Customer IDs — the field is consistent per-invoice, not randomly dropped. These invoices also look structurally distinct: far more line items per invoice (median 20 vs. 15, mean 77.6 vs. 21.2, driven by a long tail of large assortment orders like Invoice 537434's 675 lines) but lower quantity per line (3.04 vs. 13.53) and a smaller revenue share than row share (13.5% of net revenue from 22.1% of rows). 98.8% UK-based. Reads as a wholesale/trade channel, not technical data loss.
- **Decision**: No rows dropped, no placeholder ID imputed. Full table used for product/time/basket-level analyses; a documented `Customer_ID.notna()` subset (~77.3% of rows) will be used for customer-level analyses (RFM, segmentation, churn, CLV) when those notebooks are built.
- **Reasoning**: Dropping these rows would erase real business activity, including one of the dataset's largest single orders. But customer-level analysis genuinely requires a customer to attribute behavior to — there's no way around that honestly.
- **Rows affected**: 0 removed, 0 new columns — a documented decision, not a transformation.

## Step 7: Standardized Description via majority vote per StockCode

In [14]:
genuine_mask = ~retail_clean['is_stock_adjustment'] & ~retail_clean['is_cancelled']

canonical_desc = (
    retail_clean[genuine_mask]
    .groupby('StockCode')['Description']
    .agg(lambda s: s.value_counts().index[0])
)

retail_clean['Description_clean'] = retail_clean['StockCode'].map(canonical_desc)
retail_clean['Description_clean'] = retail_clean['Description_clean'].fillna(retail_clean['Description'])

both_notna = retail_clean["Description"].notna() & retail_clean['Description_clean'].notna()
changed = (both_notna & (retail_clean['Description_clean'] != retail_clean['Description'])).sum()
filled = (retail_clean['Description'].isna() & retail_clean['Description_clean'].notna()).sum()
print(f"Rows standardized to a different wording:{changed}")
print(f"Rows where a missing Description got filled from another row of the same StockCode: {filled}")

Rows standardized to a different wording:62433
Rows where a missing Description got filled from another row of the same StockCode: 3889


### Cleaning log:
### Step 7: Standardized Description via majority vote per StockCode
- **Issue**: 646 of 4,907 product codes (13.2%) had more than one Description; 8 had more than 3. Down from 1,230/75 before excluding adjustment/cancellation rows, confirming most of the apparent problem was warehouse-note contamination already handled in Steps 4–5.
- **Evidence**: Case/whitespace normalization alone only resolved 25 of the 646 remaining cases — the real pattern is naming drift over time (e.g., "POLKADOT" vs "RETROSPOT" vs "WHITE SPOT" used for the same print across StockCodes 22344/22345/22346/22384), not typos.
- **Action**: Added `Description_clean`, built as the most frequent Description per StockCode among genuine (non-adjustment, non-cancelled) sales rows, applied to all rows. Original `Description` untouched.
- **Reasoning**: StockCode remains the true identifier for every quantitative calculation, so this step is purely about reporting quality — consistent labels for dashboards, Market Basket Analysis output, and the Products table — not about correctness of any number.
- **Known limitation**: for the 8 severe cases, majority vote assumes all variants refer to one product. One code (23236) mixes "STORAGE TIN VINTAGE DOILY" (192 rows) with "DOILEY BISCUIT TIN" (13 rows) — possibly a genuinely different item sharing a reused code, not just a naming variant. Left as-is given the volume (13 of ~1M rows); flagged here rather than silently resolved.
- **Rows affected**: 0 removed, 1 column added.

## Step 8: Flagged and corrected extreme Price outliers


In [15]:
'''genuine_mask = ~retail_clean['is_stock_adjustment'] & ~retail_clean['is_cancelled']
median_price = retail_clean[genuine_mask].groupby('StockCode')['Price'].median()

retail_clean['median_price_for_code'] = retail_clean['StockCode'].map(median_price)
price_ratio = retail_clean['Price'] / retail_clean['median_price_for_code'].replace(0, np.nan)

retail_clean['is_price_outlier'] = price_ratio > 10

retail_clean['Price_clean'] = retail_clean['Price'].where(~retail_clean['is_price_outlier'], retail_clean['median_price_for_code'])

print(f"Flagged as price outliers: {retail_clean['is_price_outlier'].sum()}")'''


genuine_mask = ~retail_clean['is_stock_adjustment'] & ~retail_clean['is_cancelled']
median_price = retail_clean[genuine_mask].groupby('StockCode')['Price'].median()

retail_clean['median_price_for_code'] = retail_clean['StockCode'].map(median_price)
price_ratio = retail_clean['Price'] / retail_clean['median_price_for_code'].replace(0, np.nan)

# Restricted to genuine rows — a stock-adjustment row's Price=0 is intentional
# (Step 4's decision), not something this step should touch or overwrite
retail_clean['is_price_outlier'] = genuine_mask & (price_ratio > 10)

retail_clean['Price_clean'] = retail_clean['Price'].where(~retail_clean['is_price_outlier'], retail_clean['median_price_for_code'])

print(f"Flagged as price outliers: {retail_clean['is_price_outlier'].sum()}")

Flagged as price outliers: 744


### Cleaning log:
### Step 8: Flagged and corrected extreme Price outliers
- **Issue**: 744 rows (0.07%) have a Price over 10x their own product's median among genuine sales — e.g., StockCode 84016 sells at £0.42 in 47 of 61 rows, but also shows six single, non-repeating entries up to £1,157.15.
- **Investigation**: Tested whether these were line-totals mistaken for unit prices (Price ÷ Quantity ≈ normal price) — held for only 7 of 929 candidates, rejected as a general cause. Distributions show two shapes: scattered non-repeating outliers (likely genuine entry errors, e.g. 84016) vs. a smaller but consistent second cluster (possibly a real second pricing tier, e.g. 37410) — not reliably distinguishable across 303 codes automatically, so flagged rows are corrected uniformly and conservatively.
- **Distinguished from**: 185 low-ratio rows, NOT flagged — nearly all are the same £0 promotional items already identified and kept in Step 4; re-flagging them here would reverse that decision.
- **Action**: Added `is_price_outlier` (>10x StockCode median) and `Price_clean` (median substituted only where flagged). Original `Price` untouched.
- **Reasoning**: Median substitution is conservative and reversible — it stops a handful of rows from distorting revenue or RFM Monetary scores, without discarding the transaction itself (Quantity, customer, date all stay usable).
- **Rows affected**: 744 flagged and corrected in `Price_clean`. 0 rows removed.

### Step 9 — saving the cleaned dataset 

1. Select and order the final columns — drop median_price_for_code, since it was scratch work for computing is_price_outlier/Price_clean and isn't meaningful on its own:

In [16]:
output_cols = [
    'Invoice', 'StockCode', 'Description','Description_clean', 'Quantity', 
    'InvoiceDate', 'Price', 'Price_clean', 'Customer_ID','Country', 'is_cancelled', 
    'is_stock_adjustment', 'original_was_cancelled', 
    'exclude_from_demand', 'is_price_outlier'
]
final = retail_clean[output_cols].copy()

2. Sort chronologically — gives a predictable default order and a natural starting point for Week 2's time-series forecasting:

In [17]:
final = final.sort_values('InvoiceDate').reset_index(drop=True)

3. Validate before writing anything — this file becomes the single source of truth for every phase after this; a silent mistake here surfaces as a confusing bug three weeks from now, far from its actual cause:

In [18]:
assert len(final) == 1_027_614, f"Unexpected row count: {len(final)}"
assert final.duplicated().sum() == 0, "Unexpected duplicates remain"
for col in ['Invoice','StockCode','Quantity','InvoiceDate','Price','Price_clean','Country']:
    assert final[col].isna().sum() == 0, f"Unexpected nulls in {col}"
print("All validation checks passed.")

All validation checks passed.


4. Save — Parquet as primary, CSV kept but git-ignored:

In [19]:
import os
os.makedirs("../data/processed", exist_ok=True)

final.to_parquet("../data/processed/online_retail_II_cleaned.parquet", index=False)
final.to_csv("../data/processed/online_retail_II_cleaned.csv", index=False)

print(f"Parquet: {os.path.getsize('../data/processed/online_retail_II_cleaned.parquet')/1e6:.1f} MB")
print(f"CSV: {os.path.getsize('../data/processed/online_retail_II_cleaned.csv')/1e6:.1f} MB")

FileNotFoundError: [WinError 2] The system cannot find the file specified: '../data/processed/online_retail_cleaned.parquet'

In [ ]:
print("Rows:", len(retail_clean))
print("Stock adjustments:", retail_clean["is_stock_adjustment"].sum())
print("Cancellations:", retail_clean["is_cancelled"].sum())
print("Missing Customer IDs:", retail_clean["Customer_ID"].isna().sum())

Rows: 1027614
Stock adjustments: 5929
Cancellations: 17923
Missing Customer IDs: 233116


In [ ]:
print("Genuine rows:", genuine_mask.sum())
print("Price ratio > 10:", (price_ratio > 10).sum())
print("Price ratio < 0.1:", (price_ratio < 0.1).sum())

Genuine rows: 1003762
Price ratio > 10: 752
Price ratio < 0.1: 5748


## Step 3 – Missing Value Analysis

In [ ]:
retail_clean.isnull().sum()

Invoice                        0
StockCode                      0
Description                 4265
Quantity                       0
InvoiceDate                    0
Price                          0
Customer_ID               233116
Country                        0
is_stock_adjustment            0
is_cancelled                   0
original_was_cancelled         0
exclude_from_demand            0
Description_clean            376
median_price_for_code        423
is_price_outlier               0
Price_clean                    0
Revenue                        0
dtype: int64

In [ ]:
(retail_clean.isnull().sum()/len(retail_clean))*100

Invoice                    0.000000
StockCode                  0.000000
Description                0.415039
Quantity                   0.000000
InvoiceDate                0.000000
Price                      0.000000
Customer_ID               22.685172
Country                    0.000000
is_stock_adjustment        0.000000
is_cancelled               0.000000
original_was_cancelled     0.000000
exclude_from_demand        0.000000
Description_clean          0.036590
median_price_for_code      0.041163
is_price_outlier           0.000000
Price_clean                0.000000
Revenue                    0.000000
dtype: float64

### Observation

Most columns contain no missing values. Missing values are primarily concentrated in the Customer_ID column, while Description contains a very small number of missing records.

These values will be evaluated before deciding whether to remove or retain them.

## Step 4 – Duplicate Records

In [ ]:
'''duplicates =retail_clean[retail_clean.duplicated()]
duplicates.shape'''


'duplicates =retail_clean[retail_clean.duplicated()]\nduplicates.shape'

### Step 5 — Data Types

In [ ]:
'''retail_clean["InvoiceDate"] = pd.to_datetime(
    retail_clean["InvoiceDate"]
)
retail_clean["InvoiceDate"].head(10)'''

'retail_clean["InvoiceDate"] = pd.to_datetime(\n    retail_clean["InvoiceDate"]\n)\nretail_clean["InvoiceDate"].head(10)'

In [ ]:
retail_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1027614 entries, 0 to 1027613
Data columns (total 17 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   Invoice                 1027614 non-null  str           
 1   StockCode               1027614 non-null  str           
 2   Description             1023349 non-null  str           
 3   Quantity                1027614 non-null  int64         
 4   InvoiceDate             1027614 non-null  datetime64[us]
 5   Price                   1027614 non-null  float64       
 6   Customer_ID             794498 non-null   Int64         
 7   Country                 1027614 non-null  str           
 8   is_stock_adjustment     1027614 non-null  bool          
 9   is_cancelled            1027614 non-null  bool          
 10  original_was_cancelled  1027614 non-null  bool          
 11  exclude_from_demand     1027614 non-null  bool          
 12  Description_clean       1

### Step 6 — Negative Quantity

In [ ]:
retail_clean[retail_clean["Quantity"] < 0].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,is_stock_adjustment,is_cancelled,original_was_cancelled,exclude_from_demand,Description_clean,median_price_for_code,is_price_outlier,Price_clean,Revenue
175,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia,False,True,False,True,PAPER BUNTING WHITE LACE,2.95,False,2.95,-35.4
176,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia,False,True,False,True,CREAM FELT EASTER EGG BASKET,1.65,False,1.65,-9.9
177,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia,False,True,False,True,POTTING SHED SOW 'N' GROW SET,4.25,False,4.25,-17.0
178,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia,False,True,False,True,POTTING SHED TWINE,2.10,False,2.10,-12.6
179,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia,False,True,False,True,PAPER CHAIN KIT RETROSPOT,2.95,False,2.95,-35.4


In [ ]:
(retail_clean["Quantity"] <0).sum()

np.int64(21316)

### Negative Price 

In [ ]:
(retail_clean["Price"]<0).sum()

np.int64(0)

### Zero Quantity

In [ ]:
(retail_clean["Quantity"] == 0).sum()

np.int64(0)

### Zero Price

In [ ]:
(retail_clean["Price"] == 0).sum()

np.int64(5990)

### Cancelled Invoices

In [ ]:
retail_clean["Invoice"].str.startswith("C").sum()

np.int64(17923)

In [ ]:
retail_clean[retail_clean["Invoice"].str.startswith("C")].head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,is_stock_adjustment,is_cancelled,original_was_cancelled,exclude_from_demand,Description_clean,median_price_for_code,is_price_outlier,Price_clean,Revenue
175,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia,False,True,False,True,PAPER BUNTING WHITE LACE,2.95,False,2.95,-35.4
176,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia,False,True,False,True,CREAM FELT EASTER EGG BASKET,1.65,False,1.65,-9.9
177,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia,False,True,False,True,POTTING SHED SOW 'N' GROW SET,4.25,False,4.25,-17.0
178,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia,False,True,False,True,POTTING SHED TWINE,2.10,False,2.10,-12.6
179,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia,False,True,False,True,PAPER CHAIN KIT RETROSPOT,2.95,False,2.95,-35.4


### Step 11 — Revenue

In [ ]:
retail_clean["Revenue"] = (retail_clean["Quantity"]*retail_clean["Price"])